# ML System Design Interview Framework

A universal, reusable template for any ML system design interview question.

**Goal**: Given an open-ended prompt ("Design a system that ..."), produce a structured, end-to-end design in 40-50 minutes that demonstrates breadth *and* depth.

---

## High-Level Flow

```
1. Problem Clarification  (5 min)
2. Metrics Definition     (5 min)
3. Data                   (5 min)
4. Feature Engineering    (5 min)
5. Model Selection        (5 min)
6. Training Pipeline      (5 min)
7. Serving & Inference    (5 min)
8. Monitoring & Iteration (5 min)
```

Not every problem needs equal time at each stage.  Lean in where complexity lives (e.g., fraud detection is heavy on features and monitoring; recs are heavy on serving and ranking).

---
## Stage 1 — Problem Clarification

**Why it matters**: Interviewers intentionally leave the prompt vague. Asking good clarifying questions signals seniority.

### What to ask

| Dimension | Example Questions |
|-----------|------------------|
| **Scope** | What exactly are we optimizing? Is this a ranking, classification, generation, or retrieval problem? |
| **Users / Scale** | How many users? How many items? How many requests per second at peak? |
| **Latency** | Is this real-time (p99 < 100ms) or is batch acceptable? |
| **Constraints** | Budget, team size, timeline. Existing infrastructure? (e.g., already on AWS, already have Spark) |
| **Business Context** | Why now? What is the current solution? What improvement are we targeting? |
| **Edge Cases** | Cold start, new users, new items, adversarial behavior |

### Estimation tips

Quick back-of-envelope calculations anchor the design:

- **QPS**: DAU x avg_requests_per_user / 86400. Peak ~ 3-5x average.
- **Storage**: num_records x avg_record_size_bytes. 1 TB = 10^12 bytes.
- **Model size**: dense layer params = input_dim x output_dim + output_dim (bias). Embedding table = vocab_size x embed_dim x 4 bytes (float32).
- **Training data**: think in terms of label volume. 1M labeled examples is a reasonable starting point for deep models.

### Common follow-ups
- "What if we had 10x the users?" (tests scalability thinking)
- "What if we only have 1 week of data?" (tests cold-start / bootstrapping)
- "What if latency budget is cut in half?" (tests serving optimization)

---
## Stage 2 — Metrics

Always define metrics *before* talking about models. This shows you think about the problem, not just the solution.

### Offline Metrics (measured during development)

| Metric | When to Use | Notes |
|--------|------------|-------|
| **AUC-ROC** | Binary classification (CTR, fraud) | Threshold-independent, but can be misleading with heavy class imbalance |
| **AUC-PR** | Imbalanced classification | Better than ROC when positives are rare |
| **Log Loss** | Calibration matters (ads, bidding) | Penalizes confident wrong predictions |
| **NDCG@k** | Ranking (search, recs) | Accounts for position and graded relevance |
| **Recall@k** | Retrieval stage | Did we retrieve the relevant item in top-k? |
| **MAP** | Ranking with binary relevance | Mean average precision |
| **BLEU / ROUGE** | Text generation | BLEU for precision, ROUGE for recall |
| **RMSE / MAE** | Regression (ETA, pricing) | MAE more robust to outliers |

### Online Metrics (measured in production)

| Metric | When to Use |
|--------|------------|
| **CTR** | Ads, recommendations, search |
| **Conversion Rate** | E-commerce, signup flows |
| **Revenue / GMV** | Ads, marketplace |
| **Engagement** (time spent, sessions) | Content platforms |
| **Retention** (D1, D7, D30) | Long-term health |

### Guardrail Metrics

These must *not* degrade even if the primary metric improves:

- Latency (p50, p99)
- Error rate
- Fairness metrics (demographic parity, equalized odds)
- Content quality / safety scores
- User complaints, reports

### Common follow-ups
- "Your offline metric improves but online doesn't — what happened?" (distribution shift, offline-online gap, proxy metric mismatch)
- "How do you handle conflicting metrics?" (multi-objective optimization, scalarization, Pareto front)
- "How long do you run an A/B test?" (power analysis, minimum detectable effect, novelty effects)

---
## Stage 3 — Data

### Data Sources

Think about *implicit* vs *explicit* signals:

| Type | Examples | Pros | Cons |
|------|---------|------|------|
| **Explicit** | Ratings, reviews, surveys | High signal | Sparse, biased (selection bias) |
| **Implicit** | Clicks, dwell time, purchases, skips | Abundant | Noisy, confounded by position bias |
| **Side information** | User profile, item metadata, context | Always available | May be stale or incomplete |

### Labeling Strategies

1. **Natural labels**: clicks, purchases, conversions (cheapest, but delayed and noisy)
2. **Human annotation**: expensive but high quality; use for eval sets and hard cases
3. **Weak supervision**: heuristic labeling functions (Snorkel-style), regex, distant supervision
4. **Semi-supervised**: self-training, co-training, label propagation
5. **Active learning**: query the most informative samples for human labeling

### Data Pipeline

```
Raw Events → ETL → Feature Store → Training Data
                  ↘ Streaming   → Real-time Features
```

Key concerns: data freshness, schema evolution, backfill capability, point-in-time correctness (avoid label leakage).

### Storage

- **Raw logs**: S3 / GCS (cheap, append-only)
- **Structured features**: data warehouse (BigQuery, Redshift, Snowflake)
- **Feature store**: Feast, Tecton, or in-house (Redis for online, Hive/S3 for offline)
- **Embeddings**: vector DB (Pinecone, Milvus, FAISS index on disk)

### Common follow-ups
- "How do you handle missing data?" (imputation, indicator features, model-based)
- "How do you ensure training-serving consistency?" (feature store, logged features)
- "How do you handle PII?" (anonymization, differential privacy, access controls)

---
## Stage 4 — Feature Engineering

### Raw → Features

| Raw Signal | Feature Ideas |
|-----------|---------------|
| Timestamp | hour_of_day, day_of_week, is_weekend, time_since_last_action |
| Text | TF-IDF, n-grams, embeddings (BERT, sentence-transformers), length, language |
| Categorical | one-hot, target encoding, hashing, embedding lookup |
| Numerical | binning, log transform, z-score, min-max, interaction terms |
| Sequence | aggregates (count, mean, std over windows), last-N items, session features |
| Graph | node degree, PageRank, community membership, GNN embeddings |

### Feature Store Architecture

```
Offline Store (batch features):
  - Historical features computed on Spark/Flink
  - Stored in data warehouse / Parquet on S3
  - Used for training data generation

Online Store (real-time features):
  - Precomputed features materialized to Redis/DynamoDB
  - Low-latency lookup at serving time
  - Must match offline computation exactly (training-serving skew)

Real-time features (computed at request time):
  - Features that depend on the current request (query text, time-of-day)
  - Streaming aggregates (clicks in last 5 min) via Flink/Kafka Streams
```

### Key Principles
- **Training-serving skew** is the #1 silent killer. Always log the features used at serving time and compare against training features.
- **Feature importance** analysis (SHAP, permutation importance) guides iteration.
- **Feature freshness**: some features need sub-second freshness (real-time auction), others can be daily.

### Common follow-ups
- "How do you handle high-cardinality categoricals?" (hashing, embedding, frequency cutoff)
- "How do you detect feature drift?" (PSI — Population Stability Index, KS test)
- "How do you handle features that are expensive to compute?" (caching, precomputation, feature selection)

---
## Stage 5 — Model Selection

### The Progression (always start simple)

```
Heuristic / Rules
  → Logistic Regression / Linear Model
    → Gradient Boosted Trees (XGBoost, LightGBM)
      → Neural Networks (MLP, DCN, Transformers)
        → Ensemble / Stacking
```

### Tradeoff Matrix

| Model | Interpretability | Latency | Data Efficiency | Handles Non-linearity |
|-------|-----------------|---------|----------------|----------------------|
| Logistic Regression | High | Very Low | High | No (manual features) |
| GBDT | Medium (SHAP) | Low | Medium | Yes |
| DNN | Low | Medium-High | Low (needs lots of data) | Yes |
| Transformer | Low | High | Low | Yes |

### When to use what

- **Logistic Regression**: first baseline, when interpretability is critical, when features are well-engineered
- **GBDT**: tabular data, medium-scale, when you need a strong baseline fast
- **DNN**: large-scale, when you have embeddings, when feature interactions are complex
- **Two-stage**: retrieval (ANN + simple model) → ranking (complex model). Almost always used for recs and search.

### Multi-objective

Many real systems optimize multiple objectives:
- Scalarization: `final_score = w1 * p_click + w2 * p_purchase + w3 * p_engagement`
- Multi-task learning: shared bottom layers, task-specific heads (MMoE, PLE)
- Constrained optimization: maximize engagement subject to quality >= threshold

### Common follow-ups
- "Why not just use a transformer for everything?" (latency, data requirements, diminishing returns on tabular)
- "How do you decide when to move from GBDT to DNN?" (scale of data, need for embedding features, interaction complexity)
- "How do you handle model fairness?" (demographic parity constraints, adversarial debiasing, post-hoc calibration)

---
## Stage 6 — Training Pipeline

### Batch vs Online Learning

| Aspect | Batch Training | Online Learning |
|--------|---------------|----------------|
| Frequency | Daily / Weekly | Continuous (per-event or micro-batch) |
| Data | Full historical dataset | Streaming events |
| Use when | Stable distribution | Fast-changing patterns (ads, trending) |
| Complexity | Lower | Higher (need to handle concept drift, catastrophic forgetting) |
| Rollback | Easy (just re-deploy previous model) | Hard (state is incremental) |

### Retraining Strategy

1. **Scheduled retraining**: retrain daily/weekly on a sliding window of data
2. **Triggered retraining**: retrain when performance drops below threshold
3. **Continuous learning**: update model weights with each new batch (warm-start)

### Validation Strategy

- **Temporal split** (not random!): train on data before time T, validate on data after T
- **Rolling window**: train on [t-30d, t-1d], validate on [t, t+1d], slide forward
- **Stratified**: for imbalanced problems, ensure class distribution is preserved
- **Cross-validation**: k-fold for smaller datasets, but temporal CV for time-series

### Training Infrastructure

- **Data parallel**: split data across GPUs, synchronize gradients (DDP in PyTorch)
- **Model parallel**: split model layers across GPUs (for very large models)
- **Pipeline parallel**: split model stages across GPUs, overlap computation
- **Hyperparameter tuning**: Bayesian optimization (Optuna), grid/random search, population-based training

### Common follow-ups
- "How do you detect that the model needs retraining?" (monitoring offline metrics on fresh data, PSI on features)
- "How do you handle catastrophic forgetting in online learning?" (replay buffer, EWC, progressive nets)
- "How do you version models and data?" (MLflow, DVC, model registry with metadata)

---
## Stage 7 — Serving & Inference

### Batch vs Real-time Inference

| Aspect | Batch | Real-time |
|--------|-------|----------|
| Latency | Minutes-hours | <100ms |
| Use case | Email recs, daily reports | Search, ad auction |
| Infra | Spark job, cron | Model server (TF Serving, Triton, TorchServe) |
| Cost | Lower (off-peak) | Higher (always-on, autoscaling) |

### Model Compression / Optimization

1. **Quantization**: FP32 → FP16 / INT8 (2-4x speedup, minimal quality loss)
2. **Distillation**: train a smaller student model to mimic the teacher
3. **Pruning**: remove small-magnitude weights (structured > unstructured for actual speedup)
4. **Architecture search**: find smaller architectures (NAS, MobileNet-style)
5. **Caching**: cache predictions for popular queries/items (huge win in practice)
6. **Feature precomputation**: compute expensive features offline, store in feature store

### A/B Testing

- **Randomization unit**: typically user-level (not request-level, to avoid inconsistent experience)
- **Sample size**: power analysis — need enough users to detect MDE (minimum detectable effect)
- **Duration**: at least 1-2 weeks to capture weekly patterns; watch for novelty/primacy effects
- **Ramp-up**: 1% → 5% → 20% → 50% → 100%. Monitor guardrails at each step.
- **Interleaving**: for ranking, interleave results from control and treatment (more sensitive than A/B)

### Serving Architecture (typical)

```
Request → API Gateway → Feature Service → Model Server → Post-processing → Response
                             ↓
                      Feature Store (Redis)
```

### Common follow-ups
- "How do you handle model rollback?" (blue-green deployment, canary, shadow mode)
- "How do you reduce tail latency?" (timeout with fallback, hedged requests, precomputation)
- "How do you handle the cold start for A/B tests?" (multi-armed bandits, Thompson sampling)

---
## Stage 8 — Monitoring & Feedback Loops

### What to Monitor

| Layer | What to Track |
|-------|---------------|
| **Data** | Input distribution shifts (PSI, KS test), missing values, schema changes |
| **Features** | Feature drift, coverage, null rates, outliers |
| **Model** | Prediction distribution, calibration, offline metric on live data |
| **System** | Latency (p50, p95, p99), throughput, error rate, memory/CPU |
| **Business** | Online KPIs (CTR, revenue, retention), user complaints |

### Data Drift Detection

- **Population Stability Index (PSI)**: compares training vs serving feature distributions
  - PSI < 0.1: no shift
  - 0.1 < PSI < 0.25: moderate shift (investigate)
  - PSI > 0.25: significant shift (retrain)
- **KL divergence**, **KS test**, **chi-squared test** for different feature types

### Model Decay

Models degrade over time because the world changes:
- User behavior shifts (seasonality, trends, external events)
- New items / entities not seen in training (cold start)
- Adversarial adaptation (fraud, spam)
- Feature pipeline breakage (silent failures)

### Alerting Strategy

```
Severity Levels:
  P0 (page): Model serving errors > 1%, latency > 5x baseline
  P1 (alert): Significant metric degradation (>5% relative drop)
  P2 (ticket): Feature drift detected, model staleness
  P3 (dashboard): Minor distribution shifts, logging anomalies
```

### Feedback Loops

**Positive feedback loop** (dangerous): model recommends popular items → they get more clicks → model reinforces popularity. Mitigation: exploration, diversity injection, popularity bias correction.

**Negative feedback loop** (useful): model detects spam → removes it → less spam in training data → model must adapt. Mitigation: keep negative examples, counterfactual logging.

### Common follow-ups
- "How do you debug a sudden metric drop?" (check data pipeline first, then feature distributions, then model predictions, then business context)
- "How do you handle feedback loops in recommendations?" (exploration, inverse propensity scoring, off-policy evaluation)
- "How do you do online evaluation without hurting users?" (shadow mode, interleaving, small holdout)

---
## Example Walkthrough: "Design a system to rank posts in a social media feed"

Let us apply the full template to a concrete problem.

### 1. Problem Clarification

- **Scope**: Rank posts (text, image, video) for a user's home feed
- **Scale**: 500M DAU, ~50 feed refreshes/day → ~25B ranking requests/day → ~290K QPS average, ~1M peak
- **Latency**: < 200ms end-to-end for feed generation
- **Current solution**: reverse-chronological feed (no ML)
- **Goal**: increase engagement (time spent, likes, shares) without degrading user satisfaction

### 2. Metrics

- **Offline**: NDCG (ranking quality), AUC for individual engagement predictions
- **Online**: time spent, like rate, share rate, DAU retention
- **Guardrails**: content quality score, misinformation rate, diverse content exposure

### 3. Data

- Implicit signals: impressions, clicks, likes, shares, comments, dwell time, hides, reports
- User profiles: demographics, interests, social graph
- Post metadata: author, content type, text, image embeddings, timestamp
- Label: weighted engagement (0.1 * click + 0.3 * like + 0.5 * share + 0.1 * comment - 1.0 * hide)

### 4. Feature Engineering

- **User features**: historical engagement rates, active hours, preferred content types
- **Post features**: author popularity, content embedding, freshness, media type
- **Cross features**: user-author affinity (interaction history), content-preference match
- **Context**: time of day, device type, network speed

### 5. Model

- **Retrieval** (1000 candidates from millions): lightweight model + heuristic filters (recency, social graph)
- **Ranking** (score 1000 → top 50): multi-task DNN predicting P(click), P(like), P(share), P(hide)
- **Re-ranking**: apply diversity rules, deduplicate authors, enforce content policies
- Final score: `w1*P(click) + w2*P(like) + w3*P(share) - w4*P(hide)`

### 6. Training

- Daily retraining on last 30 days of data (temporal split)
- Multi-task learning with shared embedding layers
- Negative sampling: show-but-no-engage as negatives

### 7. Serving

- Feature store (Redis) for precomputed user/post features
- Model server (TF Serving / Triton) with GPU inference
- Cache top-K for repeat feed pulls within a session
- A/B testing at user level, 2-week minimum

### 8. Monitoring

- Track engagement rates per content type daily
- Monitor prediction calibration weekly
- Alert on >3% drop in any engagement metric
- Feedback loop mitigation: inject exploration candidates (10% random), measure content diversity

---
## Quick Reference Estimation Cheat Sheet

In [ ]:
# ============================================================
# Quick estimation helpers for system design interviews
# ============================================================

def estimate_qps(dau: float, requests_per_user_per_day: float, peak_factor: float = 3.0):
    """Estimate queries per second."""
    avg_qps = dau * requests_per_user_per_day / 86400
    peak_qps = avg_qps * peak_factor
    print(f"DAU: {dau:,.0f}")
    print(f"Requests/user/day: {requests_per_user_per_day}")
    print(f"Average QPS: {avg_qps:,.0f}")
    print(f"Peak QPS ({peak_factor}x): {peak_qps:,.0f}")
    return avg_qps, peak_qps


def estimate_storage(num_records: float, avg_record_bytes: float):
    """Estimate storage in human-readable units."""
    total_bytes = num_records * avg_record_bytes
    units = [("PB", 1e15), ("TB", 1e12), ("GB", 1e9), ("MB", 1e6), ("KB", 1e3)]
    for name, threshold in units:
        if total_bytes >= threshold:
            print(f"Records: {num_records:,.0f}")
            print(f"Avg record size: {avg_record_bytes:,.0f} bytes")
            print(f"Total storage: {total_bytes / threshold:,.1f} {name}")
            return total_bytes
    print(f"Total storage: {total_bytes:,.0f} bytes")
    return total_bytes


def estimate_model_params(layers: list):
    """Estimate dense model parameters.
    layers: list of (input_dim, output_dim) tuples.
    """
    total = 0
    for i, (inp, out) in enumerate(layers):
        params = inp * out + out  # weights + bias
        total += params
        print(f"Layer {i}: {inp} -> {out} = {params:,} params")
    size_fp32 = total * 4
    size_fp16 = total * 2
    print(f"\nTotal params: {total:,}")
    print(f"Size (FP32): {size_fp32 / 1e6:.1f} MB")
    print(f"Size (FP16): {size_fp16 / 1e6:.1f} MB")
    return total


def estimate_embedding_table(vocab_size: int, embed_dim: int):
    """Estimate embedding table size."""
    params = vocab_size * embed_dim
    size_bytes = params * 4  # float32
    print(f"Vocab: {vocab_size:,}  Embed dim: {embed_dim}")
    print(f"Params: {params:,}")
    print(f"Size (FP32): {size_bytes / 1e9:.2f} GB")
    return params

In [ ]:
# Example: Social media feed ranking
print("=== Feed Ranking QPS ===")
estimate_qps(dau=500e6, requests_per_user_per_day=50)

print("\n=== Daily Log Storage ===")
# 500M users * 50 requests * 2KB per log entry
estimate_storage(num_records=500e6 * 50, avg_record_bytes=2000)

print("\n=== Ranking Model Size ===")
estimate_model_params([
    (512, 256),   # input layer
    (256, 128),   # hidden 1
    (128, 64),    # hidden 2
    (64, 4),      # output (P(click), P(like), P(share), P(hide))
])

print("\n=== User Embedding Table ===")
estimate_embedding_table(vocab_size=500_000_000, embed_dim=64)

---
## Summary: Interview Checklist

Before you finish your design, verify:

- [ ] Clarified the problem scope and constraints
- [ ] Defined offline AND online metrics, plus guardrails
- [ ] Identified data sources and labeling strategy
- [ ] Described key features and how they are computed (batch vs real-time)
- [ ] Justified model choice (started simple, explained when to upgrade)
- [ ] Addressed training: frequency, validation, infrastructure
- [ ] Addressed serving: latency, throughput, compression, A/B testing
- [ ] Addressed monitoring: drift, decay, alerting, feedback loops
- [ ] Discussed tradeoffs at each stage (not just the happy path)
- [ ] Prepared for "what if" follow-ups (scale, failure modes, edge cases)